# 15 — Fusion Retrieval (RAG Fusion)

> **Tier 3 | Query Handling**

## The Problem

A single query embedding captures only one angle of a topic. Different phrasings retrieve different relevant chunks:

```
Q: "What causes sea level rise?"

Variant A: "Factors driving ocean level increase"
→ retrieves passages about thermal expansion

Variant B: "Ice sheet and glacier contribution to rising seas"
→ retrieves passages about meltwater from Greenland/Antarctica

Variant C: "Consequences of higher ocean levels for coastlines"
→ retrieves passages about flooding and displacement
```

Any single phrasing leaves relevant chunks on the table.

## Fusion Retrieval Solution

Generate **N query variants** with an LLM, run a retrieval pass per variant,
then merge all ranked lists using **Reciprocal Rank Fusion (RRF)**:

```
RRF_score(doc) = Σ  1 / (k + rank_i(doc))    k = 60
```

Documents that rank highly across multiple variants rise to the top; documents
unique to one variant still contribute proportionally.

## Flow Diagram

```mermaid
flowchart TD
    Q(["❓ Original Question"])

    Q --> VG["🧠 LLM Variant Generator\n→ variant 1 … variant N-1"]

    subgraph RETRIEVAL ["🔍  Per-Variant Retrieval"]
        direction LR
        V0["Original\n→ top-K hits"]
        V1["Variant 1\n→ top-K hits"]
        VN["Variant N-1\n→ top-K hits"]
    end

    Q  --> V0
    VG --> V1
    VG --> VN

    V0 --> RRF["🔀 Reciprocal Rank Fusion\nRRF_score = Σ 1/(k + rank_i)"]
    V1 --> RRF
    VN --> RRF

    RRF --> TOP["✂️ Top-K fused chunks"]
    TOP --> SYNTH["🟠 Strands Agent\nGenerate answer"]
    SYNTH --> ANS(["✅ Broad, recall-maximised answer"])

    style RETRIEVAL fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style RRF       fill:#dcfce7,stroke:#16a34a,color:#14532d
    style SYNTH     fill:#ffedd5,stroke:#f97316,color:#7c2d12
```

## Step 1 — Install Dependencies

In [1]:
import subprocess, sys
packages = ["boto3","qdrant-client","opensearch-py","requests-aws4auth",
            "strands-agents","pypdf","python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")

All packages ready.


## Step 2 — Imports & Configuration

In [2]:
import os, json, time, uuid, re
from typing import List, Dict, Optional
from collections import defaultdict

import boto3, pypdf
from dotenv import load_dotenv
from strands import Agent
from strands.models.bedrock import BedrockModel
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue
)

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)

AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

COLLECTION_NAME  = "fusion_retrieval_15"
EMBEDDING_DIM    = 1024
CHUNK_SIZE       = 500
CHUNK_OVERLAP    = 50
PDF_PATH         = r"C:\Users\Administrator\RAG\data\climate.pdf"

# Fusion-specific knobs
NUM_VARIANTS     = 4    # original + 3 LLM-generated variants
TOP_K_PER_QUERY  = 8    # candidates fetched per variant
FINAL_TOP_K      = 5    # passages kept after RRF
RRF_K            = 60   # rank-dampening constant (standard default)

print(f"Collection  : {COLLECTION_NAME}")
print(f"PDF exists  : {os.path.exists(PDF_PATH)}")
print(f"AWS region  : {AWS_REGION}")
print(f"Key ID      : {os.getenv('AWS_ACCESS_KEY_ID','NOT SET')[:12]}...")
print(f"Variants    : {NUM_VARIANTS}  |  Top-K/variant : {TOP_K_PER_QUERY}  |  Final top-K : {FINAL_TOP_K}")

Collection  : fusion_retrieval_15
PDF exists  : True
AWS region  : us-east-1
Key ID      : ASIA4KPWEP6M...
Variants    : 4  |  Top-K/variant : 8  |  Final top-K : 5


## Step 3 — Vector Store

In [3]:
class VectorStore:
    def __init__(self, collection_name, qdrant_url='', qdrant_api_key='',
                 opensearch_url='', region='us-east-1'):
        self.name = collection_name
        self._backend = None
        if qdrant_url:
            try:
                kw = {'url': qdrant_url}
                if qdrant_api_key: kw['api_key'] = qdrant_api_key
                self._qdrant = QdrantClient(**kw)
                self._qdrant.get_collections()
                self._backend = 'qdrant_cloud'
                print(f'Qdrant Cloud: {qdrant_url}')
                return
            except Exception as e:
                print(f'Qdrant Cloud unavailable ({e}), trying next...')
        if opensearch_url:
            try:
                from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
                creds = boto3.Session().get_credentials()
                auth  = AWSV4SignerAuth(creds, region, 'aoss')
                host  = opensearch_url.replace('https://','').replace('http://','')
                self._os = OpenSearch(
                    hosts=[{'host': host, 'port': 443}],
                    http_auth=auth, use_ssl=True, verify_certs=True,
                    connection_class=RequestsHttpConnection, timeout=30)
                self._os.info()
                self._backend = 'opensearch'
                print(f'OpenSearch: {opensearch_url}')
                return
            except Exception as e:
                print(f'OpenSearch unavailable ({e}), falling back...')
        self._qdrant  = QdrantClient(':memory:')
        self._backend = 'qdrant_memory'
        print('Using Qdrant in-memory')

    def create_collection(self, dim=1024, force_recreate=False):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            exists = self.name in [c.name for c in self._qdrant.get_collections().collections]
            if exists and force_recreate:
                self._qdrant.delete_collection(self.name); exists = False
            if not exists:
                self._qdrant.create_collection(
                    self.name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
                print(f'Created "{self.name}" (dim={dim})')
            else:
                print(f'"{self.name}" already exists — skipping recreate')

    def upsert(self, docs: List[Dict]) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            pts = [PointStruct(id=str(uuid.uuid4()), vector=d['embedding'],
                               payload={'text': d['text'], 'metadata': d.get('metadata', {})})
                   for d in docs]
            self._qdrant.upsert(collection_name=self.name, points=pts)
            return len(pts)

    def search(self, qvec: List[float], top_k: int = 5,
               query_filter=None) -> List[Dict]:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            resp = self._qdrant.query_points(
                collection_name=self.name, query=qvec, limit=top_k,
                query_filter=query_filter, with_payload=True)
            return [{'text': p.payload.get('text', ''),
                     'metadata': p.payload.get('metadata', {}),
                     'score': p.score, 'id': str(p.id)}
                    for p in resp.points]
        return []

    def count(self) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            return self._qdrant.get_collection(self.name).points_count or 0
        return 0

    def delete_collection(self):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            self._qdrant.delete_collection(self.name)
        print(f'Deleted "{self.name}"')

print("VectorStore defined.")

VectorStore defined.


## Step 4 — Bedrock Helpers

In [4]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

_model = BedrockModel(model_id=LLM_MODEL, region_name=AWS_REGION)

def llm(prompt: str) -> str:
    return str(Agent(model=_model, system_prompt="You are a helpful assistant.")(prompt))

test_emb = embed_text("fusion retrieval test")
print(f"Embedding OK — dim={len(test_emb)}")
print("Bedrock LLM ready.")

Embedding OK — dim=1024
Bedrock LLM ready.


## Step 5 — Connect & Index climate.pdf

In [5]:
def recursive_split(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    separators = ["\n\n", "\n", ". ", " ", ""]
    def _split(text, seps):
        sep, new_seps = '', []
        for i, s in enumerate(seps):
            if s == '' or s in text:
                sep, new_seps = s, seps[i+1:]; break
        parts = text.split(sep) if sep != '' else list(text)
        good = []
        for part in parts:
            if len(part) <= chunk_size: good.append(part)
            elif new_seps: good.extend(_split(part, new_seps))
            else:
                for k in range(0, len(part), chunk_size - chunk_overlap):
                    good.append(part[k:k+chunk_size])
        chunks, cur_pieces, cur_len = [], [], 0
        for piece in good:
            p = piece.strip()
            if not p: continue
            addition = len(sep) + len(p) if cur_pieces else len(p)
            if cur_len + addition <= chunk_size:
                cur_pieces.append(p); cur_len += addition
            else:
                if cur_pieces: chunks.append(sep.join(cur_pieces))
                ov, ovl = [], 0
                for prev in reversed(cur_pieces):
                    if ovl + len(prev) + len(sep) <= chunk_overlap:
                        ov.insert(0, prev); ovl += len(prev) + len(sep)
                    else: break
                cur_pieces = ov + [p]
                cur_len = sum(len(x) + len(sep) for x in cur_pieces)
        if cur_pieces: chunks.append(sep.join(cur_pieces))
        return [c for c in chunks if c.strip()]
    return _split(text, separators)

vs = VectorStore(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL,
    qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL,
    region=AWS_REGION
)
print(f"Backend: {vs._backend}")
vs.create_collection(dim=EMBEDDING_DIM, force_recreate=True)

reader    = pypdf.PdfReader(PDF_PATH)
full_text = ''
page_map  = []
for pg_idx, page in enumerate(reader.pages):
    pg_text = (page.extract_text() or '') + '\n\n'
    page_map.extend([pg_idx + 1] * len(pg_text))
    full_text += pg_text

chunks = recursive_split(full_text, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"PDF pages: {len(reader.pages)}  |  Chunks: {len(chunks)}")

embs = embed_batch(chunks, label='[index]')
docs = []
for i, (chunk, emb) in enumerate(zip(chunks, embs)):
    char_offset = full_text.find(chunk[:40])
    page_num = page_map[min(char_offset, len(page_map)-1)] if char_offset >= 0 else 1
    docs.append({
        'text'     : chunk,
        'embedding': emb,
        'metadata' : {'chunk_idx': i, 'page_num': page_num, 'char_count': len(chunk)}
    })
vs.upsert(docs)
print(f"Indexed {vs.count()} vectors into '{COLLECTION_NAME}'")

Qdrant Cloud: https://2210ff1c-7c49-4fec-91f4-01662586299c.us-east-1-1.aws.cloud.qdrant.io
Backend: qdrant_cloud
Created "fusion_retrieval_15" (dim=1024)
PDF pages: 13  |  Chunks: 66
  [index] 20/66
  [index] 40/66
  [index] 60/66
Indexed 66 vectors into 'fusion_retrieval_15'


## Step 6 — Query Variant Generator

The LLM produces `NUM_VARIANTS - 1` alternative phrasings of the original question,
each approaching the topic from a different angle: restatement, decomposition, or
perspective shift. The original is always included as variant 0.

In [6]:
def generate_variants(question: str, n_variants: int = NUM_VARIANTS) -> List[str]:
    n_extra = n_variants - 1
    prompt = (
        f"Generate exactly {n_extra} alternative phrasings of the following question.\n"
        f"Each variant should approach the same topic from a different angle:\n"
        f"  - Try rephrasing, broadening, narrowing, or shifting perspective\n"
        f"  - Keep each variant a single, self-contained question\n"
        f"  - Output ONLY a numbered list, one variant per line, no extra text\n\n"
        f"Original question: {question}\n\nVariants:"
    )
    raw = llm(prompt).strip()
    variants = []
    for line in raw.splitlines():
        cleaned = re.sub(r'^\d+[.)\s]+', '', line.strip()).strip()
        if cleaned:
            variants.append(cleaned)
    return [question] + variants[:n_extra]

# Demo
demo_q = "What causes global temperature increase?"
variants = generate_variants(demo_q)
print(f"Original  : {variants[0]}")
for i, v in enumerate(variants[1:], 1):
    print(f"Variant {i} : {v}")

1. How do human activities and natural processes contribute to the warming of Earth's climate?
2. What specific greenhouse gases are most responsible for trapping heat in the atmosphere?
3. In what ways has the rise in global temperatures affected weather patterns and ecosystems worldwide?Original  : What causes global temperature increase?
Variant 1 : How do human activities and natural processes contribute to the warming of Earth's climate?
Variant 2 : What specific greenhouse gases are most responsible for trapping heat in the atmosphere?
Variant 3 : In what ways has the rise in global temperatures affected weather patterns and ecosystems worldwide?


## Step 7 — Reciprocal Rank Fusion (RRF)

Merge N ranked result lists into one using the RRF formula:

```
RRF_score(doc) = Σ_i  1 / (k + rank_i(doc))
```

- Documents appearing near the top of **multiple** lists score highest
- Rank-based (not score-based) — robust to scale differences across queries
- `k = 60` dampens the outsized effect of very-top ranks (empirically validated)

In [7]:
def reciprocal_rank_fusion(results_list: List[List[Dict]],
                            k: int = RRF_K) -> List[Dict]:
    rrf_scores  = defaultdict(float)
    appearances = defaultdict(int)
    doc_store   = {}

    for results in results_list:
        for rank, doc in enumerate(results, start=1):
            doc_id = doc['id']
            rrf_scores[doc_id]  += 1.0 / (k + rank)
            appearances[doc_id] += 1
            if doc_id not in doc_store:
                doc_store[doc_id] = doc

    fused = []
    for doc_id, score in sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True):
        entry = dict(doc_store[doc_id])
        entry['rrf_score']   = round(score, 6)
        entry['appearances'] = appearances[doc_id]
        fused.append(entry)
    return fused

# Unit test — doc B appears in both lists and should rank first
r1 = [{'id':'a','text':'A','metadata':{},'score':0.9},
      {'id':'b','text':'B','metadata':{},'score':0.8},
      {'id':'c','text':'C','metadata':{},'score':0.6}]
r2 = [{'id':'b','text':'B','metadata':{},'score':0.95},
      {'id':'a','text':'A','metadata':{},'score':0.7},
      {'id':'d','text':'D','metadata':{},'score':0.5}]
fused_test = reciprocal_rank_fusion([r1, r2])
print("RRF unit test (B in both lists → should rank 1st):")
for i, r in enumerate(fused_test, 1):
    print(f"  {i}. id={r['id']}  rrf={r['rrf_score']:.5f}  appearances={r['appearances']}")

RRF unit test (B in both lists → should rank 1st):
  1. id=a  rrf=0.03252  appearances=2
  2. id=b  rrf=0.03252  appearances=2
  3. id=c  rrf=0.01587  appearances=1
  4. id=d  rrf=0.01587  appearances=1


## Step 8 — Full Fusion RAG Pipeline

End-to-end:
1. Generate `NUM_VARIANTS` query variants via LLM
2. Embed each variant with Titan V2
3. Retrieve `TOP_K_PER_QUERY` chunks per variant
4. Merge all result lists with RRF → keep `FINAL_TOP_K`
5. Synthesise answer with Strands Agent

In [8]:
def rag_fusion(question: str,
               num_variants: int = NUM_VARIANTS,
               top_k_per_query: int = TOP_K_PER_QUERY,
               final_top_k: int = FINAL_TOP_K,
               verbose: bool = True) -> Dict:
    t0 = time.time()

    # 1. Generate query variants
    variants = generate_variants(question, num_variants)

    # 2. Retrieve for each variant
    all_results = []
    for v in variants:
        qvec = embed_text(v)
        hits = vs.search(qvec, top_k=top_k_per_query)
        all_results.append(hits)

    # 3. Fuse
    fused = reciprocal_rank_fusion(all_results)[:final_top_k]

    # 4. Build context
    context_parts = []
    for h in fused:
        m   = h['metadata']
        src = f"[p.{m.get('page_num','?')}]"
        context_parts.append(f"{src}\n{h['text']}")
    context = '\n\n'.join(context_parts)

    # 5. Synthesise
    synth_prompt = (
        "Answer the question using ONLY the context below. "
        "Cite page numbers (e.g. [p.3]) for key facts. "
        "If the answer is not in the context, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    )
    answer  = llm(synth_prompt)
    latency = (time.time() - t0) * 1000

    if verbose:
        print(f"\nQuestion: {question}")
        print(f"Variants ({len(variants)}):")
        for i, v in enumerate(variants):
            tag = '(original)' if i == 0 else f'(variant {i})'
            print(f"  {tag}: {v}")
        print(f"\nFused top-{final_top_k} chunks:")
        for i, r in enumerate(fused, 1):
            pg  = r['metadata'].get('page_num','?')
            rrf = r['rrf_score']
            app = r['appearances']
            print(f"  [{i}] p.{pg}  rrf={rrf:.5f}  in {app}/{len(variants)} lists  "
                  f"{r['text'][:70]}...")
        print(f"\nAnswer:\n{answer}")
        print(f"\nLatency: {latency:.0f}ms")
        print('-' * 70)

    return {
        'question'  : question,
        'variants'  : variants,
        'fused'     : fused,
        'answer'    : answer,
        'latency_ms': latency,
    }

# Demo
rag_fusion("What causes global temperature increase?")

1. How do human activities and natural processes contribute to the warming of Earth's climate?
2. What specific greenhouse gases are most responsible for trapping heat in the atmosphere?
3. In what ways has industrialization changed the balance of energy entering and leaving the Earth's surface?The provided context does not contain information about the causes of global temperature increase. The context focuses on weather analysis, forecasting, data collection methods, and the WMO's weather monitoring network, but does not address the causes of global temperature increase.
Question: What causes global temperature increase?
Variants (4):
  (original): What causes global temperature increase?
  (variant 1): How do human activities and natural processes contribute to the warming of Earth's climate?
  (variant 2): What specific greenhouse gases are most responsible for trapping heat in the atmosphere?
  (variant 3): In what ways has industrialization changed the balance of energy entering 

{'question': 'What causes global temperature increase?',
 'variants': ['What causes global temperature increase?',
  "How do human activities and natural processes contribute to the warming of Earth's climate?",
  'What specific greenhouse gases are most responsible for trapping heat in the atmosphere?',
  "In what ways has industrialization changed the balance of energy entering and leaving the Earth's surface?"],
 'fused': [{'text': 'Melbourne in Australia and Moscow in Russia. This enables in the dissemination of daily\nweather forecasts and early and reliable warnings of high-impact weather and climate events.\nAfter the processes of recording, collection and transmission of data comes the work of\ncompilation and analysis of data which is done by climatological experts. Computers are used in\nthe final analysis work and various models are in use by the experts which we would learn in the',
   'metadata': {'chunk_idx': 17, 'page_num': 5, 'char_count': 461},
   'score': 0.1483581,
 

## Step 9 — Comparison: Simple RAG vs Fusion RAG

Run both pipelines on the same questions and compare page coverage.
Extra pages found by fusion = recall gain from multi-variant retrieval.

In [9]:
def rag_simple(question: str, top_k: int = FINAL_TOP_K) -> Dict:
    qvec    = embed_text(question)
    hits    = vs.search(qvec, top_k=top_k)
    context = '\n\n'.join(f"[p.{h['metadata'].get('page_num','?')}]\n{h['text']}" for h in hits)
    answer  = llm(
        f"Answer using ONLY the context. Cite page numbers.\n\n"
        f"Context:\n{context}\n\nQ: {question}\n\nA:"
    )
    return {'hits': hits, 'answer': answer}

test_questions = [
    "What are the projected sea level rise figures for 2100?",
    "How do greenhouse gas emissions affect ocean chemistry?",
    "What evidence shows changes in extreme weather frequency?",
    "How does Arctic sea ice loss affect global climate?",
]

print(f"{'Question':<52}  {'Simple pages':>14}  {'Fusion pages':>14}  {'Extra':>8}")
print('-' * 96)

for q in test_questions:
    r_simple = rag_simple(q)
    r_fusion = rag_fusion(q, verbose=False)

    simple_pages = set(h['metadata'].get('page_num','?') for h in r_simple['hits'])
    fusion_pages = set(h['metadata'].get('page_num','?') for h in r_fusion['fused'])
    extra        = fusion_pages - simple_pages

    print(f"{q[:51]:<52}  {str(sorted(simple_pages)):>14}  "
          f"{str(sorted(fusion_pages)):>14}  {str(sorted(extra)):>8}")

Question                                                Simple pages    Fusion pages     Extra
------------------------------------------------------------------------------------------------
The provided context does not contain any information about projected sea level rise figures for 2100. The context only discusses weather forecasting methods, forecast accuracy, and related topics.1. How much could global sea levels increase by the end of the 21st century according to current climate models?
2. Which coastal regions face the greatest risk from ocean level changes expected to occur before 2100?
3. What factors do scientists consider when estimating how high the world's oceans will rise over the next several decades?The context provided does not contain any information about projected sea level rise figures for 2100. The passages focus on weather forecasting methods, including long-range forecasts, trends methods, and cyclone prediction techniques.What are the projected sea level ri

## Step 10 — Full Demo on a Broad Question

A broad question is where multi-variant retrieval helps most — the exact phrasing
matters less when RRF aggregates across several angles.

In [10]:
rag_fusion(
    "What are the main drivers and projected consequences of climate change "
    "for ecosystems and human populations?",
    num_variants=4,
    top_k_per_query=8,
    final_top_k=6
)

1. How do greenhouse gas emissions and other human activities accelerate climate change, and what long-term effects can ecosystems and societies expect as a result?
2. In what ways are rising global temperatures already reshaping biodiversity, wildlife habitats, and the daily lives of communities around the world?
3. Which specific environmental and socioeconomic factors make certain ecosystems and human populations most vulnerable to the impacts of a changing climate?Based on the provided context, **I cannot answer this question**. The context does not contain information about the main drivers or projected consequences of climate change for ecosystems and human populations.

The context touches briefly on weather forecasting methods, the World Meteorological Organization (WMO), and types of weather forecasts [p.8, p.9, p.11], but does not address climate change drivers (such as greenhouse gas emissions) or their projected consequences for ecosystems and human populations. You would n

{'question': 'What are the main drivers and projected consequences of climate change for ecosystems and human populations?',
 'variants': ['What are the main drivers and projected consequences of climate change for ecosystems and human populations?',
  'How do greenhouse gas emissions and other human activities accelerate climate change, and what long-term effects can ecosystems and societies expect as a result?',
  'In what ways are rising global temperatures already reshaping biodiversity, wildlife habitats, and the daily lives of communities around the world?',
  'Which specific environmental and socioeconomic factors make certain ecosystems and human populations most vulnerable to the impacts of a changing climate?'],
 'fused': [{'text': 'day-to-day activities, for example in aviation, transport, tourism, sports, health, adventure\nactivities and for managing the disasters. This is because such forecasts are weather specific\nand predict specific weather phenomenon like fog, thunde

## Step 11 — Summary

| Component | Implementation |
|-----------|---------------|
| Variant generator | Claude Sonnet 4.6 → N-1 alternative phrasings |
| Retrieval | Top-K vector search per variant (independent) |
| Fusion | Reciprocal Rank Fusion (k=60) |
| Context order | RRF-ranked (cross-list consensus first) |
| Synthesis | Single LLM call over fused context |
| Vector DB | Qdrant Cloud → OpenSearch → in-memory |
| Embeddings | Amazon Bedrock Titan V2 (1024-dim) |

## How Fusion Retrieval differs from Query Decomposition (nb 13)

| | Query Decomposition | Fusion Retrieval |
|---|---|---|
| Goal | Cover distinct sub-topics | Cover one topic from multiple angles |
| Variants | Semantically different sub-questions | Rephrasing of the same question |
| Merging | Dedup by chunk ID | Reciprocal Rank Fusion (rank-weighted) |
| Best for | Multi-part questions | Broad or ambiguous single questions |
| Extra LLM calls | 1 decompose | 1 variant-gen |

## How Fusion Retrieval differs from Step-Back Prompting (nb 14)

| | Step-Back Prompting | Fusion Retrieval |
|---|---|---|
| Direction | Zooms OUT from specific to general | Stays at same level, N phrasings |
| Retrieval passes | 2 (specific + step-back) | N (one per variant) |
| Merging | Simple dedup; step-back first | RRF weighted by cross-list rank |
| Best for | Specific questions needing background | Broad/ambiguous questions |

## When to use Fusion Retrieval

- User query is ambiguous or phrased in domain jargon the corpus may not use
- Document vocabulary is heterogeneous (same concept named differently)
- Recall matters more than latency (4× more embedding calls vs simple RAG)
- Questions starting with *what are the main …*, *how does … relate to …*

### Next: **16 — Chain-of-Thought RAG**

In [11]:
# vs.delete_collection()  # uncomment to clean up
print(f"Collection '{COLLECTION_NAME}' in {vs._backend} — {vs.count()} vectors")
print("\nDone. Give the go-ahead for notebook 16.")

Collection 'fusion_retrieval_15' in qdrant_cloud — 66 vectors

Done. Give the go-ahead for notebook 16.
